**NYC Taxi Trips - Basics**

**Dataset**: samples.nyctaxi.trips

**Difficulty**: Easy

**Topics:** filter, count, aggregation, sorting

In [0]:
from pyspark.sql import functions as f, types as t

Learn — Loading, Filtering, and Aggregating
- Function	What it does
- spark.table("catalog.schema.table")	Loads a Delta/Unity Catalog table into a DataFrame
- df.count()	Triggers compute and returns the total number of rows (action)
- df.filter(condition)	Returns rows matching the condition (transformation — lazy)
- df.groupBy(col).agg(...)	Groups rows by a column and applies aggregate functions
- df.orderBy(col.desc())	Sorts rows in descending order
- F.col("name")	References a column by name
- F.avg(), F.min(), F.max(), F.count()	Aggregation functions

In [0]:
df=spark.table("samples.nyctaxi.trips")

print("total rows", df.count())
#filter to long trips (over 5 miles)
filtered_df=df.filter(f.col("trip_distance")>5)

#compute trip stats by pickup ZIP code
df.groupBy("pickup_zip").agg(f.count("*").alias("total_trips"), f.round(f.avg("fare_amount"),2).alias("avg_fare")).orderBy(f.col("total_trips").desc()).show()

**Problem 1**

Count the total number of trips in the NYC taxi dataset. Load the table samples.nyctaxi.trips and return a single-row DataFrame that tells us how many records exist.

Expected output columns:

total_trips (bigint) - total number of trip records

In [0]:
df.printSchema()


In [0]:
df=spark.read.table("samples.nyctaxi.trips")

result_1=df.agg(f.count("*").alias("total_trips"))
result_1.show()

In [0]:

# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'total_trips' in cols, "Missing column: total_trips"
assert len(cols) == 1, f"Expected exactly 1 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt == 1, f"Expected exactly 1 row, got {cnt}"
total = result_1.collect()[0]['total_trips']
assert total > 0, f"Expected total_trips > 0, got {total}"
assert total < 10_000_000, f"total_trips seems unreasonably large: {total}"
print(f"Problem 1 passed ✓  ({cnt} rows, total_trips={total})")

**Problem 2**

Find all trips where the trip distance is greater than 10 miles. Return the full original row for each qualifying trip.

Expected output columns:

- tpep_pickup_datetime - pickup timestamp
- tpep_dropoff_datetime - dropoff timestamp
- trip_distance - distance in miles (must be > 10)
- fare_amount - fare charged
- pickup_zip - pickup ZIP code
- dropoff_zip - dropoff ZIP code

In [0]:
result_2 = df.filter(f.col("trip_distance")>10)

In [0]:

assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'tpep_pickup_datetime' in cols, "Missing column: tpep_pickup_datetime"
assert 'tpep_dropoff_datetime' in cols, "Missing column: tpep_dropoff_datetime"
assert 'trip_distance' in cols, "Missing column: trip_distance"
assert 'fare_amount' in cols, "Missing column: fare_amount"
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
assert 'dropoff_zip' in cols, "Missing column: dropoff_zip"
assert len(cols) == 6, f"Expected exactly 6 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_dist = result_2.agg(f.min('trip_distance')).collect()[0][0]
assert min_dist > 10, f"All trip_distance values must be > 10, found min={min_dist}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

**Problem 3**

Calculate summary statistics for the fare amount across all trips. Return a single-row DataFrame with the average, minimum, and maximum fare.

Expected output columns:

- avg_fare - average fare amount across all trips
- min_fare - minimum fare amount
- max_fare - maximum fare amount

In [0]:
result_3=df.agg(f.avg("fare_amount").alias("avg_fare"), f.min("fare_amount").alias("min_fare"), f.max("fare_amount").alias("max_fare"))

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'avg_fare' in cols, "Missing column: avg_fare"
assert 'min_fare' in cols, "Missing column: min_fare"
assert 'max_fare' in cols, "Missing column: max_fare"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt == 1, f"Expected exactly 1 row, got {cnt}"
row = result_3.collect()[0]
assert row['max_fare'] >= row['avg_fare'] >= row['min_fare'], "Expected min_fare <= avg_fare <= max_fare"
print(f"Problem 3 passed ✓  ({cnt} rows)")

**Problem 4**

Find the top 5 pickup ZIP codes by the number of trips originating there. Sort the results in descending order of trip count.

Expected output columns:

- pickup_zip - the ZIP code where trips started
- trip_count - number of trips from that ZIP code (sorted descending)

In [0]:
result_4=df.groupBy("pickup_zip").agg(f.count("*").alias("trip_count")).orderBy(f.col("trip_count").desc()).limit(5)
result_4.show()

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
assert 'trip_count' in cols, "Missing column: trip_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt == 5, f"Expected exactly 5 rows (top 5), got {cnt}"
counts = [r['trip_count'] for r in result_4.collect()]
assert counts == sorted(counts, reverse=True), "Results must be sorted by trip_count descending"
assert all(c > 0 for c in counts), "All trip counts must be positive"
print(f"Problem 4 passed ✓  ({cnt} rows)")

**Problem 5**

Count the number of trips per dropoff ZIP code, but only include trips where the fare amount is greater than $20. This helps identify which areas attract higher-value trips.

Expected output columns:

- dropoff_zip - the ZIP code where trips ended
- trip_count - number of qualifying trips (fare > $20) ending at that ZIP